<a href="https://colab.research.google.com/github/Carolaynebarret/DataConnect/blob/main/Mini_Desafio_DataConnec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Carregamento dos Dados e Função de Diagnóstico

Primeiro, vamos importar as bibliotecas necessárias, definir os caminhos dos arquivos e criar uma função auxiliar para adivinhar a chave primária de um DataFrame, que será usada para verificar duplicatas por chave. Em seguida, definimos a função principal `diagnose_dataframe` que executa todas as verificações solicitadas para um único DataFrame.

In [ ]:
import pandas as pd
import numpy as np

# Caminhos dos arquivos CSV
file_paths = {
    'apontamentos': '/content/dados/apontamentos.csv',
    'projetos': '/content/dados/projetos.csv',
    'clientes': '/content/dados/clientes.csv',
    'analistas': '/content/dados/analistas.csv',
    'satisfacao': '/content/dados/satisfacao.csv',
}

# Dicionário para armazenar os DataFrames
dfs = {}

# Função auxiliar para tentar adivinhar a chave primária de um DataFrame
def guess_primary_key(df_name, df_columns):
    potential_keys = [
        f'{df_name}_id',
        'id',
        f'cd_{df_name}', # Considerando a menção de 'dc_' no prompt
        f'{df_name}Id',
        'ID'
    ]
    for pk in potential_keys:
        if pk in df_columns:
            return pk
    return None

def diagnose_dataframe(df_name, df):
    print(f"\n{'='*50}\n--- Diagnóstico para: {df_name.upper()} ---\n{'='*50}")

    # 1. Quantas linhas e colunas
    print(f"\n1. Dimensões do DataFrame: {df.shape[0]} linhas, {df.shape[1]} colunas")

    # 2. Nomes das colunas
    print("\n2. Nomes das Colunas:")
    for col in df.columns:
        print(f"  - {col}")

    # 3. 10 primeiras linhas
    print("\n3. 10 Primeiras Linhas:")
    display(df.head(10))

    # 4. Tipos de cada coluna
    print("\n4. Tipos de Dados das Colunas (`df.info()`):")
    df.info()

    # 5. Nulos por coluna
    null_counts = df.isnull().sum()
    print("\n5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):")
    if null_counts.sum() > 0:
        display(null_counts[null_counts > 0])
    else:
        print("  Não há valores nulos em nenhuma coluna.")

    # 6. Linhas duplicadas
    full_duplicates = df.duplicated().sum()
    print(f"\n6. Linhas Duplicadas Completas: {full_duplicates}")

    # Duplicatas pela chave primária
    pk = guess_primary_key(df_name, df.columns)
    if pk:
        pk_duplicates = df.duplicated(subset=[pk]).sum()
        print(f"   Duplicatas pela chave primária '{pk}': {pk_duplicates}")
        if pk_duplicates > 0:
            print(f"   Exemplos de linhas duplicadas pela chave '{pk}':")
            display(df[df.duplicated(subset=[pk], keep=False)].sort_values(by=pk).head())
    else:
        print(f"   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou '{df_name}_id') para '{df_name}'. Não foi possível verificar duplicatas por chave.")

    # 7. Valores únicos de colunas categóricas e espaços em branco
    print("\n7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):")
    object_cols = df.select_dtypes(include='object').columns
    if object_cols.empty:
        print("  Não há colunas do tipo 'object' para analisar.")
    else:
        for col in object_cols:
            print(f"\n  - Coluna '{col}':")
            unique_vals = df[col].astype(str).unique()
            if len(unique_vals) <= 50: # Mostrar valores únicos se não forem muitos
                print(f"    Valores únicos ({len(unique_vals)}): {unique_vals.tolist()}")
            else:
                print(f"    {len(unique_vals)} valores únicos (mostrando os 5 primeiros): {unique_vals[:5].tolist()}...")

            # Verificar espaços em branco no início/fim
            has_leading_trailing_spaces = df[col].astype(str).str.contains(r'^\s|\s$', na=False).any()
            if has_leading_trailing_spaces:
                print(f"    AVISO: Existem valores com espaços em branco no início ou fim nesta coluna. Exemplo: '{df[col][df[col].astype(str).str.contains(r'^\s|\s$', na=False)].iloc[0]}'")
            else:
                print("    Não foram encontrados valores com espaços em branco no início ou fim.")

    # 8. Datas: formatos inconsistentes
    print("\n8. Análise de Colunas de Data:")
    potential_date_cols = df.select_dtypes(include='object').columns.tolist() + df.select_dtypes(include=['datetime64']).columns.tolist()
    found_inconsistent_date = False
    for col in potential_date_cols:
        # Tentar converter para datetime, forçando erros para NaT
        temp_date_series = pd.to_datetime(df[col], errors='coerce')

        # Se houver NaTs introduzidos, mas não todas as linhas, há inconsistência
        if temp_date_series.isnull().sum() > 0 and temp_date_series.isnull().sum() < len(df[col]):
            print(f"  AVISO: Coluna '{col}' contém datas com formatos inconsistentes ou valores não-data.\n    Valores que não puderam ser convertidos: {df[col][temp_date_series.isnull()].unique().tolist()}")
            found_inconsistent_date = True
        elif temp_date_series.isnull().sum() == 0 and df[col].dtype == 'object' and not df[col].empty:
             print(f"  Colunua '{col}' (tipo object) parece conter datas válidas e foi convertida com sucesso (temp).")
             found_inconsistent_date = True # It was successfully parsed from object, so format is 'consistent' for now
        elif temp_date_series.isnull().sum() == len(df[col]) and not df[col].empty:
            print(f"  AVISO: Coluna '{col}' (tipo object) parece não conter datas válidas, todos os valores falharam na conversão.")
            found_inconsistent_date = True

    if not found_inconsistent_date and not potential_date_cols:
        print("  Nenhuma coluna de data potencial identificada.")
    elif not found_inconsistent_date:
        print("  Todas as colunas de data identificadas parecem ter formatos consistentes ou já são do tipo datetime.")

    # 9. Outliers
    print("\n9. Análise de Outliers (Exemplos Específicos e Estatísticas Descritivas):")
    numeric_cols = df.select_dtypes(include=np.number).columns
    if numeric_cols.empty:
        print("  Não há colunas numéricas para análise de outliers.")
    else:
        print("  Estatísticas descritivas para colunas numéricas (incluindo quartis para detecção simples de outliers):")
        display(df[numeric_cols].describe())

        # Verificações específicas de outliers
        if df_name == 'apontamentos' and 'horas_apontadas' in df.columns:
            outliers_apontamentos = df[df['horas_apontadas'] > 24]
            if not outliers_apontamentos.empty:
                print(f"\n  AVISO: '{df_name}' - Existem {len(outliers_apontamentos)} apontamentos com mais de 24 horas (`horas_apontadas` > 24):")
                display(outliers_apontamentos.head())
            else:
                print(f"\n  '{df_name}' - Nenhum apontamento com mais de 24 horas encontrado em `horas_apontadas`.")

        if df_name == 'projetos' and 'valor_contrato' in df.columns:
            # Detecção de outliers usando IQR para 'valor_contrato'
            Q1 = df['valor_contrato'].quantile(0.25)
            Q3 = df['valor_contrato'].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            lower_bound = Q1 - 1.5 * IQR
            outliers_contrato = df[(df['valor_contrato'] < lower_bound) | (df['valor_contrato'] > upper_bound)]
            if not outliers_contrato.empty:
                print(f"\n  AVISO: '{df_name}' - Existem {len(outliers_contrato)} projetos com `valor_contrato` fora dos limites IQR (possíveis outliers):")
                display(outliers_contrato.head())
            else:
                print(f"\n  '{df_name}' - Nenhuma anomalia de `valor_contrato` detectada pelo método IQR simples.")

    print(f"\n{'='*50}\n--- Fim do Diagnóstico para: {df_name.upper()} ---\n{'='*50}\n")

### 2. Executando o Diagnóstico para Cada Arquivo

Agora, vamos carregar cada um dos 5 arquivos CSV para DataFrames e aplicar a função `diagnose_dataframe` que acabamos de definir. Isso nos dará uma visão detalhada do estado de cada conjunto de dados individualmente.

In [ ]:
# Carregar cada arquivo e executar o diagnóstico
for name, path in file_paths.items():
    try:
        dfs[name] = pd.read_csv(path)
        diagnose_dataframe(name, dfs[name])
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{path}' não encontrado. Verifique o caminho.\n")
    except Exception as e:
        print(f"ERRO ao processar o arquivo '{path}': {e}\n")


--- Diagnóstico para: APONTAMENTOS ---

1. Dimensões do DataFrame: 300 linhas, 5 colunas

2. Nomes das Colunas:
  - apontamento_id
  - projeto_id
  - analista_id
  - data
  - horas

3. 10 Primeiras Linhas:


,apontamento_id,projeto_id,analista_id,data,horas
0,1,101,7,2026-05-06,5.1
1,2,101,5,2026-04-15,3.3
2,3,101,7,2026-04-08,5.5
3,4,101,5,2026-04-26,3.3
4,5,101,5,2026-04-30,5.8
5,6,101,5,2026-04-10,7.4
6,7,101,7,2026-04-10,5.1
7,8,101,7,2026-04-29,4.9
8,9,101,5,2026-05-01,7.6
9,10,101,7,2026-04-15,6.7



4. Tipos de Dados das Colunas (`df.info()`):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   apontamento_id  300 non-null    int64  
 1   projeto_id      300 non-null    int64  
 2   analista_id     300 non-null    int64  
 3   data            300 non-null    object 
 4   horas           300 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 11.8+ KB

5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):
  Não há valores nulos em nenhuma coluna.

6. Linhas Duplicadas Completas: 0
   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou 'apontamentos_id') para 'apontamentos'. Não foi possível verificar duplicatas por chave.

7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):

  - Coluna 'data':
    213 valores únicos (mostrando os 5 primeiros): ['2026-05-06', '2026-0

,apontamento_id,projeto_id,analista_id,horas
count,300.000000,300.000000,300.000000,300.000000
mean,150.500000,112.406667,5.346667,5.144000
std,86.746758,6.763661,2.520781,1.742331
min,1.000000,101.000000,1.000000,2.000000
25%,75.750000,107.000000,3.000000,3.575000
50%,150.500000,112.000000,5.000000,5.300000
75%,225.250000,118.000000,8.000000,6.600000
max,300.000000,124.000000,10.000000,8.000000



--- Fim do Diagnóstico para: APONTAMENTOS ---


--- Diagnóstico para: PROJETOS ---

1. Dimensões do DataFrame: 24 linhas, 8 colunas

2. Nomes das Colunas:
  - projeto_id
  - cliente_id
  - nome
  - tipo_servico
  - data_inicio
  - data_fim
  - status
  - receita

3. 10 Primeiras Linhas:


...

,satisfacao_id,projeto_id,data,nota,comentario
0,1,101,2026-05-16,7.5,Atendeu o combinado
1,2,103,2025-10-27,9.1,Time muito claro na comunicação
2,3,105,2025-08-24,7.7,Atendeu o combinado
3,4,108,2025-05-26,8.1,Prazo apertado mas entregue
4,5,109,2026-08-17,8.9,Entrega excelente
5,6,111,2025-09-07,9.6,Superou a expectativa
6,7,114,2026-09-08,6.8,Prazo apertado mas entregue
7,8,115,2026-03-30,6.9,Prazo apertado mas entregue
8,9,116,2026-04-08,5.1,Prazo estourou
9,10,117,2026-11-12,10.0,Time muito claro na comunicação



4. Tipos de Dados das Colunas (`df.info()`):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   satisfacao_id  16 non-null     int64  
 1   projeto_id     16 non-null     int64  
 2   data           16 non-null     object 
 3   nota           16 non-null     float64
 4   comentario     16 non-null     object 
dtypes: float64(1), int64(2), object(2)
memory usage: 772.0+ bytes

5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):
  Não há valores nulos em nenhuma coluna.

6. Linhas Duplicadas Completas: 0
   Duplicatas pela chave primária 'satisfacao_id': 0

7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):

  - Coluna 'data':
    Valores únicos (16): ['2026-05-16', '2025-10-27', '2025-08-24', '2025-05-26', '2026-08-17', '2025-09-07', '2026-09-08', '2026-03-30', '2026-04-08', '2026-11-12', '2025-06-16', '2025

/tmp/ipykernel_618/776909747.py:99: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')


,satisfacao_id,projeto_id,nota
count,16.000000,16.000000,16.000000
mean,8.500000,114.000000,7.712500
std,4.760952,7.127412,1.480034
min,1.000000,101.000000,5.100000
25%,4.750000,108.750000,6.800000
50%,8.500000,115.500000,7.700000
75%,12.250000,119.250000,8.950000
max,16.000000,124.000000,10.000000



--- Fim do Diagnóstico para: SATISFACAO ---



### 3. Verificações de Consistência Entre Arquivos

Por fim, vamos realizar as verificações de consistência entre os DataFrames, garantindo que as chaves estrangeiras referenciem IDs existentes nos DataFrames relacionados.

In [ ]:
print(f"\n{'='*50}\n--- Verificações de Consistência Entre Arquivos ---\n{'='*50}")

# 1. Verificar se todo projeto_id que aparece em apontamentos existe em projetos
if 'apontamentos' in dfs and 'projetos' in dfs and 'projeto_id' in dfs['apontamentos'].columns and 'projeto_id' in dfs['projetos'].columns:
    apontamentos_projetos_ids = dfs['apontamentos']['projeto_id'].dropna().unique()
    projetos_ids = dfs['projetos']['projeto_id'].dropna().unique()
    missing_projetos_in_apontamentos = np.setdiff1d(apontamentos_projetos_ids, projetos_ids)
    if len(missing_projetos_in_apontamentos) > 0:
        print(f"\nAVISO: 'projeto_id' em `apontamentos` não encontrados em `projetos`: {len(missing_projetos_in_apontamentos)} IDs.\n  Exemplos: {missing_projetos_in_apontamentos[:5].tolist()}")
    else:
        print("\nTodos os 'projeto_id' em `apontamentos` existem em `projetos`.")
else:
    print("\nNão foi possível verificar a consistência de 'projeto_id' entre `apontamentos` e `projetos` (arquivos/colunas ausentes ou renomeadas).")

# 2. Verificar se todo cliente_id de projetos existe em clientes
if 'projetos' in dfs and 'clientes' in dfs and 'cliente_id' in dfs['projetos'].columns and 'cliente_id' in dfs['clientes'].columns:
    projetos_clientes_ids = dfs['projetos']['cliente_id'].dropna().unique()
    clientes_ids = dfs['clientes']['cliente_id'].dropna().unique()
    missing_clientes_in_projetos = np.setdiff1d(projetos_clientes_ids, clientes_ids)
    if len(missing_clientes_in_projetos) > 0:
        print(f"\nAVISO: 'cliente_id' em `projetos` não encontrados em `clientes`: {len(missing_clientes_in_projetos)} IDs.\n  Exemplos: {missing_clientes_in_projetos[:5].tolist()}")
    else:
        print("\nTodos os 'cliente_id' em `projetos` existem em `clientes`.")
else:
    print("\nNão foi possível verificar a consistência de 'cliente_id' entre `projetos` e `clientes` (arquivos/colunas ausentes ou renomeadas).")

# 3. Verificar se todo analista_id de apontamentos existe em analistas
if 'apontamentos' in dfs and 'analistas' in dfs and 'analista_id' in dfs['apontamentos'].columns and 'analista_id' in dfs['analistas'].columns:
    apontamentos_analistas_ids = dfs['apontamentos']['analista_id'].dropna().unique()
    analistas_ids = dfs['analistas']['analista_id'].dropna().unique()
    missing_analistas_in_apontamentos = np.setdiff1d(apontamentos_analistas_ids, analistas_ids)
    if len(missing_analistas_in_apontamentos) > 0:
        print(f"\nAVISO: 'analista_id' em `apontamentos` não encontrados em `analistas`: {len(missing_analistas_in_apontamentos)} IDs.\n  Exemplos: {missing_analistas_in_apontamentos[:5].tolist()}")
    else:
        print("\nTodos os 'analista_id' em `apontamentos` existem em `analistas`.")
else:
    print("\nNão foi possível verificar a consistência de 'analista_id' entre `apontamentos` e `analistas` (arquivos/colunas ausentes ou renomeadas).")

# 4. Verificar se todo projeto_id de satisfacao existe em projetos
if 'satisfacao' in dfs and 'projetos' in dfs and 'projeto_id' in dfs['satisfacao'].columns and 'projeto_id' in dfs['projetos'].columns:
    satisfacao_projetos_ids = dfs['satisfacao']['projeto_id'].dropna().unique()
    projetos_ids = dfs['projetos']['projeto_id'].dropna().unique()
    missing_projetos_in_satisfacao = np.setdiff1d(satisfacao_projetos_ids, projetos_ids)
    if len(missing_projetos_in_satisfacao) > 0:
        print(f"\nAVISO: 'projeto_id' em `satisfacao` não encontrados em `projetos`: {len(missing_projetos_in_satisfacao)} IDs.\n  Exemplos: {missing_projetos_in_satisfacao[:5].tolist()}")
    else:
        print("\nTodos os 'projeto_id' em `satisfacao` existem em `projetos`.")
else:
    print("\nNão foi possível verificar a consistência de 'projeto_id' entre `satisfacao` e `projetos` (arquivos/colunas ausentes ou renomeadas).")

print(f"\n{'='*50}\n--- Fim das Verificações de Consistência ---\n{'='*50}\n")


--- Verificações de Consistência Entre Arquivos ---

Todos os 'projeto_id' em `apontamentos` existem em `projetos`.

Todos os 'cliente_id' em `projetos` existem em `clientes`.

Todos os 'analista_id' em `apontamentos` existem em `analistas`.

Todos os 'projeto_id' em `satisfacao` existem em `projetos`.

--- Fim das Verificações de Consistência ---



### 4. Diagnóstico Detalhado para Arquivos com Saída Truncada

Como a saída anterior foi truncada, vamos agora apresentar o diagnóstico detalhado para os arquivos `projetos`, `clientes`, `analistas` e `satisfacao` individualmente para garantir que todas as informações sejam exibidas.

In [ ]:
# Re-executar diagnóstico para arquivos específicos cuja saída foi truncada
files_to_re_diagnose = ['projetos', 'clientes', 'analistas', 'satisfacao']

for name in files_to_re_diagnose:
    if name in dfs:
        diagnose_dataframe(name, dfs[name])
    else:
        print(f"ERRO: DataFrame '{name}' não encontrado no dicionário 'dfs'. Certifique-se de que o carregamento inicial foi bem-sucedido.")


--- Diagnóstico para: PROJETOS ---

1. Dimensões do DataFrame: 24 linhas, 8 colunas

2. Nomes das Colunas:
  - projeto_id
  - cliente_id
  - nome
  - tipo_servico
  - data_inicio
  - data_fim
  - status
  - receita

3. 10 Primeiras Linhas:


,projeto_id,cliente_id,nome,tipo_servico,data_inicio,data_fim,status,receita
0,101,2,ETL - Verde Agro,ETL,2026-04-08,2026-05-06,Concluído,18636.42
1,102,2,Consultoria - Verde Agro,Consultoria,2026-01-01,2026-01-26,Em andamento,32259.98
2,103,1,Consultoria - Banco Aurora,Consultoria,2025-09-14,2025-10-18,Concluído,34273.24
3,104,5,ETL - EducaBem,ETL,2025-07-11,2025-08-17,Em andamento,11500.01
4,105,6,ETL - MobiUrbana,ETL,2025-07-04,2025-08-15,Concluído,12986.46
5,106,5,ETL - EducaBem,ETL,2026-09-04,2026-10-18,Em andamento,17494.78
6,107,5,Consultoria - EducaBem,Consultoria,2026-04-23,2026-05-17,Em andamento,30990.49
7,108,4,Dashboard/BI - LojaMais Varejo,Dashboard/BI,2025-04-04,2025-05-18,Concluído,28605.38
8,109,6,SQL - MobiUrbana,SQL,2026-06-07,2026-08-08,Concluído,22508.24
9,110,2,Consultoria - Verde Agro,Consultoria,2025-09-24,2025-10-29,Em andamento,33002.06



4. Tipos de Dados das Colunas (`df.info()`):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   projeto_id    24 non-null     int64  
 1   cliente_id    24 non-null     int64  
 2   nome          24 non-null     object 
 3   tipo_servico  24 non-null     object 
 4   data_inicio   24 non-null     object 
 5   data_fim      24 non-null     object 
 6   status        24 non-null     object 
 7   receita       24 non-null     float64
dtypes: float64(1), int64(2), object(5)
memory usage: 1.6+ KB

5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):
  Não há valores nulos em nenhuma coluna.

6. Linhas Duplicadas Completas: 0
   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou 'projetos_id') para 'projetos'. Não foi possível verificar duplicatas por chave.

7. Análise de Colunas Categóricas (Valores Únicos e Espaços e

/tmp/ipykernel_618/776909747.py:99: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_618/776909747.py:99: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_618/776909747.py:99: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')


,projeto_id,cliente_id,receita
count,24.000000,24.000000,24.000000
mean,112.500000,4.291667,24746.389167
std,7.071068,1.988645,9537.604402
min,101.000000,1.000000,10981.350000
25%,106.750000,3.000000,17488.682500
50%,112.500000,4.500000,22256.275000
75%,118.250000,5.250000,32445.500000
max,124.000000,8.000000,42982.670000


...

,satisfacao_id,projeto_id,data,nota,comentario
0,1,101,2026-05-16,7.5,Atendeu o combinado
1,2,103,2025-10-27,9.1,Time muito claro na comunicação
2,3,105,2025-08-24,7.7,Atendeu o combinado
3,4,108,2025-05-26,8.1,Prazo apertado mas entregue
4,5,109,2026-08-17,8.9,Entrega excelente
5,6,111,2025-09-07,9.6,Superou a expectativa
6,7,114,2026-09-08,6.8,Prazo apertado mas entregue
7,8,115,2026-03-30,6.9,Prazo apertado mas entregue
8,9,116,2026-04-08,5.1,Prazo estourou
9,10,117,2026-11-12,10.0,Time muito claro na comunicação



4. Tipos de Dados das Colunas (`df.info()`):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   satisfacao_id  16 non-null     int64  
 1   projeto_id     16 non-null     int64  
 2   data           16 non-null     object 
 3   nota           16 non-null     float64
 4   comentario     16 non-null     object 
dtypes: float64(1), int64(2), object(2)
memory usage: 772.0+ bytes

5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):
  Não há valores nulos em nenhuma coluna.

6. Linhas Duplicadas Completas: 0
   Duplicatas pela chave primária 'satisfacao_id': 0

7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):

  - Coluna 'data':
    Valores únicos (16): ['2026-05-16', '2025-10-27', '2025-08-24', '2025-05-26', '2026-08-17', '2025-09-07', '2026-09-08', '2026-03-30', '2026-04-08', '2026-11-12', '2025-06-16', '2025

/tmp/ipykernel_618/776909747.py:99: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')


,satisfacao_id,projeto_id,nota
count,16.000000,16.000000,16.000000
mean,8.500000,114.000000,7.712500
std,4.760952,7.127412,1.480034
min,1.000000,101.000000,5.100000
25%,4.750000,108.750000,6.800000
50%,8.500000,115.500000,7.700000
75%,12.250000,119.250000,8.950000
max,16.000000,124.000000,10.000000



--- Fim do Diagnóstico para: SATISFACAO ---

